In [ ]:
import gc
import itertools
import json
import os
import queue
import subprocess
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import tkinter as tk
from tkinter import ttk, messagebox


# ============================================================
# 0. 全域設定
# ============================================================

BOARD_SIZE = 15

# 5 個指定模型
TOTAL_ITERATIONS = 1000

# 你的電腦：Ryzen 5 7600X + RTX 5070
WORKERS = 10

# 每批只派這麼多工作，避免一次建立大量 Future
CHUNK_SIZE = 100

# 每幾場存一次
BATCH_FLUSH = 100

# 每場最大時間
CPU_TURN_MS = 400
GPU_TURN_MS = 1000

CPU_MOVE_TIMEOUT = 1.2
GPU_MOVE_TIMEOUT = 2.8

CPU_GAME_MAX_TIME = 20.0
GPU_GAME_MAX_TIME = 45.0

# ============================================================
# 資料輸出資料夾 (修改為指定路徑)
# ============================================================

OUTPUT_DIR = r"C:\Users\User\共用程式\first one final\data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CSV_OVERALL = os.path.join(
    OUTPUT_DIR,
    "overall_5_tournament_ranking.csv"
)

CSV_H2H = os.path.join(
    OUTPUT_DIR,
    "overall_5_tournament_head_to_head.csv"
)

PROGRESS_FILE = os.path.join(
    OUTPUT_DIR,
    "tournament_progress.json"
)

# 每一局詳細棋步紀錄的子資料夾
MATCH_LOG_DIR = os.path.join(
    OUTPUT_DIR,
    "match_steps"
)
os.makedirs(MATCH_LOG_DIR, exist_ok=True)


# ============================================================
# 1. AI 引擎 (限定截圖中的 5 個模型)
# ============================================================

ALL_ENGINES = {

    "AlphaGomoku":
        r"C:\Users\User\共用程式\first one final\mode\AlphaGomoku_windows\pbrain-AlphaGomoku_opencl.exe",

    "VibeFive":
        r"C:\Users\User\共用程式\first one final\mode\VIBEFIVE\pbrain-vibefive.exe",

    "Rapfi":
        r"C:\Users\User\共用程式\first one final\mode\Rapfi-engine\pbrain-rapfi-windows-avx2.exe",

    "Yixin":
        r"C:\Users\User\共用程式\first one final\mode\Yixin\engine.exe",

    "PentaZen":
        r"C:\Users\User\共用程式\first one final\mode\pbrain-PentaZen21_15.exe",
}


# ============================================================
# GPU Heavy
# ============================================================

GPU_HEAVY_SET = {
    "VibeFive",
    "AlphaGomoku"
}


# ============================================================
# 2. Gomocup Engine 通訊
# ============================================================

class NonBlockingEngine:

    def __init__(
        self,
        name,
        exe_path,
        max_threads=1,
        timeout_turn_ms=600
    ):
        self.name = name
        self.exe_path = exe_path
        self.cwd = os.path.dirname(exe_path)

        self.max_threads = max_threads
        self.timeout_turn_ms = timeout_turn_ms

        self.proc = None

        self.out_queue = queue.Queue()

        self.read_thread = None

        self.alive = False

    # --------------------------------------------------------

    def _enqueue_output(self):

        while (
            self.alive
            and self.proc
            and self.proc.stdout
        ):

            try:

                line = self.proc.stdout.readline()

                if line:
                    self.out_queue.put(line)

                else:
                    break

            except Exception:

                break

    # --------------------------------------------------------

    def start(self):

        self.proc = subprocess.Popen(

            [self.exe_path],

            stdin=subprocess.PIPE,

            stdout=subprocess.PIPE,

            stderr=subprocess.DEVNULL,

            text=True,

            encoding="utf-8",

            errors="ignore",

            bufsize=1,

            cwd=self.cwd
        )

        self.alive = True

        self.read_thread = threading.Thread(
            target=self._enqueue_output,
            daemon=True
        )

        self.read_thread.start()

        self.send(
            f"INFO max_threads {self.max_threads}"
        )

        self.send(
            "INFO memory 1024"
        )

        self.send(
            f"INFO timeout_turn {self.timeout_turn_ms}"
        )

        self.send(
            f"START {BOARD_SIZE}"
        )

        self._wait_ok()

    # --------------------------------------------------------

    def send(self, cmd):

        if (
            self.proc
            and self.proc.stdin
            and self.alive
        ):

            try:

                self.proc.stdin.write(
                    cmd.strip() + "\n"
                )

                self.proc.stdin.flush()

            except Exception:

                pass

    # --------------------------------------------------------

    def _wait_ok(self):

        start_t = time.time()

        while time.time() - start_t < 3.0:

            try:

                line = self.out_queue.get(
                    timeout=0.05
                )

                if "OK" in line.upper():

                    return True

            except queue.Empty:

                pass

        return False

    # --------------------------------------------------------

    def begin(self, move_timeout):

        self.send("BEGIN")

        return self._wait_move(
            move_timeout
        )

    # --------------------------------------------------------

    def turn(
        self,
        x,
        y,
        move_timeout
    ):

        self.send(
            f"TURN {x},{y}"
        )

        return self._wait_move(
            move_timeout
        )

    # --------------------------------------------------------

    def _wait_move(self, move_timeout):

        start_t = time.time()

        while (
            time.time() - start_t
            < move_timeout
        ):

            remain = (
                move_timeout
                - (time.time() - start_t)
            )

            if remain <= 0:
                break

            try:

                line = self.out_queue.get(
                    timeout=min(
                        0.06,
                        remain
                    )
                )

            except queue.Empty:

                continue

            line = line.strip()

            if (
                line.startswith("MESSAGE")
                or
                line.startswith("DEBUG")
            ):

                continue

            if "," in line:

                try:

                    parts = line.split(",")

                    x = int(parts[0])

                    y = int(parts[1])

                    return x, y

                except ValueError:

                    continue

            if "ERROR" in line.upper():

                return None

        return None

    # --------------------------------------------------------

    def close(self):

        self.alive = False

        if self.proc:

            try:

                self.send("END")

                if self.proc.stdin:

                    self.proc.stdin.close()

                if self.proc.stdout:

                    self.proc.stdout.close()

                self.proc.terminate()

                try:

                    self.proc.wait(
                        timeout=0.2
                    )

                except subprocess.TimeoutExpired:

                    self.proc.kill()

            except Exception:

                try:

                    self.proc.kill()

                except Exception:

                    pass

            finally:

                self.proc = None

        gc.collect()


# ============================================================
# 3. 五子棋裁判
# ============================================================

class FastReferee:

    def __init__(
        self,
        size=BOARD_SIZE
    ):

        self.size = size

        self.board = [
            [0] * size
            for _ in range(size)
        ]

    # --------------------------------------------------------

    def is_valid(self, x, y):

        return (
            0 <= x < self.size
            and
            0 <= y < self.size
            and
            self.board[y][x] == 0
        )

    # --------------------------------------------------------

    def place(
        self,
        x,
        y,
        stone
    ):

        self.board[y][x] = stone

    # --------------------------------------------------------

    def check_win(
        self,
        x,
        y
    ):

        stone = self.board[y][x]

        directions = [
            (1, 0),
            (0, 1),
            (1, 1),
            (1, -1)
        ]

        for dx, dy in directions:

            count = 1

            for step in (1, -1):

                nx = x + dx * step
                ny = y + dy * step

                while (
                    0 <= nx < self.size
                    and
                    0 <= ny < self.size
                    and
                    self.board[ny][nx] == stone
                ):

                    count += 1

                    nx += dx * step
                    ny += dy * step

            if count >= 5:

                return True

        return False

    # --------------------------------------------------------

    def is_full(self):

        for row in self.board:

            for cell in row:

                if cell == 0:

                    return False

        return True


# ============================================================
# 4. 單場比賽 (包含每步紀錄)
# ============================================================

def play_match(
    b_name,
    w_name,
    b_exe,
    w_exe
):

    b_is_gpu = (
        b_name in GPU_HEAVY_SET
    )

    w_is_gpu = (
        w_name in GPU_HEAVY_SET
    )

    b_turn_ms = (
        GPU_TURN_MS
        if b_is_gpu
        else CPU_TURN_MS
    )

    w_turn_ms = (
        GPU_TURN_MS
        if w_is_gpu
        else CPU_TURN_MS
    )

    b_move_timeout = (
        GPU_MOVE_TIMEOUT
        if b_is_gpu
        else CPU_MOVE_TIMEOUT
    )

    w_move_timeout = (
        GPU_MOVE_TIMEOUT
        if w_is_gpu
        else CPU_MOVE_TIMEOUT
    )

    game_max_time = (
        GPU_GAME_MAX_TIME
        if (
            b_is_gpu
            or
            w_is_gpu
        )
        else CPU_GAME_MAX_TIME
    )

    b = NonBlockingEngine(
        b_name,
        b_exe,
        max_threads=1,
        timeout_turn_ms=b_turn_ms
    )

    w = NonBlockingEngine(
        w_name,
        w_exe,
        max_threads=1,
        timeout_turn_ms=w_turn_ms
    )

    referee = FastReferee()

    start_time = time.time()
    steps_record = []
    move_index = 1

    try:

        b.start()

        w.start()

        # 黑方先手
        move = b.begin(
            b_move_timeout
        )

        if (
            not move
            or
            not referee.is_valid(
                *move
            )
        ):
            return b_name, w_name, w_name, steps_record

        referee.place(
            move[0],
            move[1],
            1
        )

        steps_record.append({
            "Step": move_index,
            "Player": b_name,
            "Stone": "Black",
            "X": move[0],
            "Y": move[1]
        })
        move_index += 1

        if referee.check_win(
            move[0],
            move[1]
        ):
            return b_name, w_name, b_name, steps_record

        last_move = move

        turn = "white"

        while True:

            if (
                time.time()
                - start_time
                > game_max_time
            ):
                return b_name, w_name, "Draw", steps_record

            # ------------------------------------------------
            # 白方
            # ------------------------------------------------

            if turn == "white":

                move = w.turn(
                    last_move[0],
                    last_move[1],
                    w_move_timeout
                )

                player = w_name

                opponent = b_name

                stone = 2

                stone_name = "White"

                next_turn = "black"

            # ------------------------------------------------
            # 黑方
            # ------------------------------------------------

            else:

                move = b.turn(
                    last_move[0],
                    last_move[1],
                    b_move_timeout
                )

                player = b_name

                opponent = w_name

                stone = 1

                stone_name = "Black"

                next_turn = "white"

            # ------------------------------------------------
            # 無效棋步 / timeout
            # ------------------------------------------------

            if (
                not move
                or
                not referee.is_valid(
                    *move
                )
            ):
                return b_name, w_name, opponent, steps_record

            referee.place(
                move[0],
                move[1],
                stone
            )

            steps_record.append({
                "Step": move_index,
                "Player": player,
                "Stone": stone_name,
                "X": move[0],
                "Y": move[1]
            })
            move_index += 1

            # ------------------------------------------------
            # 勝利
            # ------------------------------------------------

            if referee.check_win(
                move[0],
                move[1]
            ):
                return b_name, w_name, player, steps_record

            # ------------------------------------------------
            # 和局
            # ------------------------------------------------

            if referee.is_full():
                return b_name, w_name, "Draw", steps_record

            last_move = move

            turn = next_turn

    except Exception:
        return b_name, w_name, "Draw", steps_record

    finally:

        b.close()

        w.close()


# ============================================================
# 5. 主 GUI
# ============================================================

class FullTournamentGUI:

    def __init__(self, root):

        self.root = root

        self.root.title(
            "⚡ 5模型 1000輪 5子棋高速錦標賽"
        )

        self.root.geometry(
            "1200x820"
        )

        self.root.configure(
            bg="#1e1e1e"
        )

        # ----------------------------------------------------
        # 找得到的引擎
        # ----------------------------------------------------

        self.valid_engines = {
            name: path
            for name, path
            in ALL_ENGINES.items()
            if os.path.exists(path)
        }

        self.names = list(
            self.valid_engines.keys()
        )

        # ----------------------------------------------------
        # 配對
        # ----------------------------------------------------

        self.pairs = list(
            itertools.permutations(
                self.names,
                2
            )
        )

        self.total_matches = (
            len(self.pairs)
            * TOTAL_ITERATIONS
        )

        # ----------------------------------------------------
        # 執行狀態
        # ----------------------------------------------------

        self.is_running = False

        self.pause_event = (
            threading.Event()
        )

        self.pause_event.set()

        self.stop_event = (
            threading.Event()
        )

        self.matches_done = 0

        self.start_time = None

        # ----------------------------------------------------
        # 統計
        # ----------------------------------------------------

        self.overall = {
            name: {
                "Played": 0,
                "Wins": 0,
                "Losses": 0,
                "Draws": 0
            }
            for name in self.names
        }

        self.h2h = {
            a: {
                b: {
                    "Wins": 0,
                    "Losses": 0,
                    "Draws": 0
                }
                for b in self.names
                if a != b
            }
            for a in self.names
        }

        self._build_ui()

        # 每秒更新 GUI
        self.root.after(
            1000,
            self._gui_tick
        )

    # ========================================================
    # GUI
    # ========================================================

    def _build_ui(self):

        top_frame = tk.Frame(
            self.root,
            bg="#1e1e1e"
        )

        top_frame.pack(
            fill=tk.X,
            padx=15,
            pady=10
        )

        title = tk.Label(
            top_frame,
            text="⚡ 5 模型 × 1000 輪五子棋對戰系統",
            font=(
                "Microsoft JhengHei",
                18,
                "bold"
            ),
            fg="#4da6ff",
            bg="#1e1e1e"
        )

        title.pack(
            side=tk.LEFT
        )

        self.lbl_status = tk.Label(
            top_frame,
            text="初始化中...",
            font=(
                "Microsoft JhengHei",
                10
            ),
            fg="#cccccc",
            bg="#1e1e1e"
        )

        self.lbl_status.pack(
            side=tk.RIGHT
        )

        # ----------------------------------------------------
        # 控制按鈕
        # ----------------------------------------------------

        btn_frame = tk.Frame(
            self.root,
            bg="#1e1e1e"
        )

        btn_frame.pack(
            fill=tk.X,
            padx=15,
            pady=5
        )

        self.btn_start = tk.Button(
            btn_frame,
            text="▶ 開始對戰",
            font=(
                "Microsoft JhengHei",
                11,
                "bold"
            ),
            bg="#28a745",
            fg="white",
            padx=15,
            command=self.start_tournament
        )

        self.btn_start.pack(
            side=tk.LEFT,
            padx=5
        )

        self.btn_pause = tk.Button(
            btn_frame,
            text="⏸ 暫停",
            font=(
                "Microsoft JhengHei",
                11,
                "bold"
            ),
            bg="#17a2b8",
            fg="white",
            padx=15,
            state=tk.DISABLED,
            command=self.toggle_pause
        )

        self.btn_pause.pack(
            side=tk.LEFT,
            padx=5
        )

        self.btn_stop = tk.Button(
            btn_frame,
            text="⏹ 停止並保存",
            font=(
                "Microsoft JhengHei",
                11,
                "bold"
            ),
            bg="#dc3545",
            fg="white",
            padx=15,
            state=tk.DISABLED,
            command=self.stop_tournament
        )

        self.btn_stop.pack(
            side=tk.LEFT,
            padx=5
        )

        self.btn_pick_best = tk.Button(
            btn_frame,
            text="👑 查看目前第一名",
            font=(
                "Microsoft JhengHei",
                11,
                "bold"
            ),
            bg="#ffc107",
            fg="black",
            padx=15,
            command=self.pick_champion
        )

        self.btn_pick_best.pack(
            side=tk.LEFT,
            padx=15
        )

        # ----------------------------------------------------
        # 進度條
        # ----------------------------------------------------

        self.progress_var = (
            tk.DoubleVar()
        )

        self.progress_bar = (
            ttk.Progressbar(
                self.root,
                variable=self.progress_var,
                maximum=100
            )
        )

        self.progress_bar.pack(
            fill=tk.X,
            padx=15,
            pady=5
        )

        # ----------------------------------------------------
        # 資訊
        # ----------------------------------------------------

        self.lbl_speed = tk.Label(
            self.root,
            text="速度：--",
            font=(
                "Consolas",
                11
            ),
            fg="#33ff33",
            bg="#1e1e1e"
        )

        self.lbl_speed.pack(
            anchor=tk.W,
            padx=18
        )

        self.lbl_eta = tk.Label(
            self.root,
            text="預估剩餘：--",
            font=(
                "Consolas",
                11
            ),
            fg="#33ff33",
            bg="#1e1e1e"
        )

        self.lbl_eta.pack(
            anchor=tk.W,
            padx=18
        )

        # ----------------------------------------------------
        # 排行榜
        # ----------------------------------------------------

        table_frame = tk.Frame(
            self.root,
            bg="#1e1e1e"
        )

        table_frame.pack(
            fill=tk.BOTH,
            expand=True,
            padx=15,
            pady=5
        )

        cols = (
            "Rank",
            "Model",
            "Played",
            "Wins",
            "Losses",
            "Draws",
            "WinRate"
        )

        self.tree = ttk.Treeview(
            table_frame,
            columns=cols,
            show="headings",
            height=12
        )

        headings = {
            "Rank": "排名",
            "Model": "模型",
            "Played": "總對局",
            "Wins": "勝場",
            "Losses": "敗場",
            "Draws": "和局",
            "WinRate": "勝率"
        }

        for col in cols:

            self.tree.heading(
                col,
                text=headings[col]
            )

            self.tree.column(
                col,
                anchor=tk.CENTER,
                width=110
            )

        self.tree.column(
            "Model",
            width=220,
            anchor=tk.W
        )

        self.tree.pack(
            side=tk.LEFT,
            fill=tk.BOTH,
            expand=True
        )

        scrollbar = ttk.Scrollbar(
            table_frame,
            orient=tk.VERTICAL,
            command=self.tree.yview
        )

        self.tree.configure(
            yscrollcommand=scrollbar.set
        )

        scrollbar.pack(
            side=tk.RIGHT,
            fill=tk.Y
        )

        # ----------------------------------------------------
        # Log
        # ----------------------------------------------------

        log_frame = tk.LabelFrame(
            self.root,
            text=" 實時對局播報 ",
            font=(
                "Microsoft JhengHei",
                10,
                "bold"
            ),
            fg="#ffffff",
            bg="#252526"
        )

        log_frame.pack(
            fill=tk.BOTH,
            expand=True,
            padx=15,
            pady=10
        )

        self.txt_log = tk.Text(
            log_frame,
            height=8,
            bg="#111111",
            fg="#33ff33",
            font=(
                "Consolas",
                10
            )
        )

        self.txt_log.pack(
            fill=tk.BOTH,
            expand=True,
            padx=5,
            pady=5
        )

        self._refresh_table()

        self.log(
            f"找到 {len(self.names)} 個可用模型：{', '.join(self.names)}"
        )

        self.log(
            f"總對局數：{self.total_matches:,}"
        )

    # ========================================================
    # 安全 GUI Log
    # ========================================================

    def log(self, msg):

        def _write():

            try:

                self.txt_log.insert(
                    tk.END,
                    f"{time.strftime('%H:%M:%S')} - {msg}\n"
                )

                self.txt_log.see(
                    tk.END
                )

            except Exception:

                pass

        self.root.after(
            0,
            _write
        )

    # ========================================================
    # 排行榜
    # ========================================================

    def _refresh_table(self):

        for item in self.tree.get_children():

            self.tree.delete(item)

        rows = []

        for name, data in self.overall.items():

            played = data["Played"]

            rate = (
                data["Wins"]
                / played
                * 100
                if played > 0
                else 0
            )

            rows.append({
                "Model": name,
                "Played": played,
                "Wins": data["Wins"],
                "Losses": data["Losses"],
                "Draws": data["Draws"],
                "WinRate": rate
            })

        rows.sort(
            key=lambda x: x["WinRate"],
            reverse=True
        )

        for i, row in enumerate(rows):

            tag = (
                "top1"
                if i == 0
                and row["Played"] > 0
                else "normal"
            )

            self.tree.insert(
                "",
                tk.END,
                values=(
                    i + 1,
                    row["Model"],
                    row["Played"],
                    row["Wins"],
                    row["Losses"],
                    row["Draws"],
                    f"{row['WinRate']:.2f}%"
                ),
                tags=(tag,)
            )

        self.tree.tag_configure(
            "top1",
            background="#2e4d34",
            foreground="#ffff66"
        )

    # ========================================================
    # 開始
    # ========================================================

    def start_tournament(self):

        if self.is_running:

            return

        if len(self.names) < 2:

            messagebox.showerror(
                "錯誤",
                "至少需要 2 個可用 AI 引擎。"
            )

            return

        self.is_running = True

        self.stop_event.clear()

        self.pause_event.set()

        self.start_time = time.time()

        self.btn_start.config(
            state=tk.DISABLED
        )

        self.btn_pause.config(
            state=tk.NORMAL,
            text="⏸ 暫停"
        )

        self.btn_stop.config(
            state=tk.NORMAL
        )

        self.log(
            "🚀 5模型高速錦標賽開始！"
        )

        self.log(
            f"⚡ 並發：{WORKERS} 路"
        )

        self.log(
            f"🎯 總對局：{self.total_matches:,}"
        )

        threading.Thread(
            target=self._run_loop,
            daemon=True
        ).start()

    # ========================================================
    # 暫停
    # ========================================================

    def toggle_pause(self):

        if self.pause_event.is_set():

            self.pause_event.clear()

            self.btn_pause.config(
                text="▶ 繼續"
            )

            self.log(
                "⏸ 已要求暫停；目前正在執行的對局會完成。"
            )

        else:

            self.pause_event.set()

            self.btn_pause.config(
                text="⏸ 暫停"
            )

            self.log(
                "▶ 賽事恢復。"
            )

    # ========================================================
    # 停止
    # ========================================================

    def stop_tournament(self):

        self.stop_event.set()

        self.is_running = False

        self.pause_event.set()

        self.log(
            "⏹ 正在停止派工並保存目前結果..."
        )

    # ========================================================
    # GUI 每秒更新
    # ========================================================

    def _gui_tick(self):

        try:

            if self.total_matches > 0:

                pct = (
                    self.matches_done
                    /
                    self.total_matches
                    * 100
                )

                self.progress_var.set(
                    pct
                )

            if (
                self.start_time
                and self.matches_done > 0
            ):

                elapsed = (
                    time.time()
                    - self.start_time
                )

                speed = (
                    self.matches_done
                    /
                    elapsed
                    * 3600
                )

                remaining = (
                    self.total_matches
                    - self.matches_done
                )

                if speed > 0:

                    remain_sec = (
                        remaining
                        /
                        speed
                        * 3600
                    )

                    self.lbl_speed.config(
                        text=(
                            f"速度："
                            f"{speed:,.0f} 場/小時"
                        )
                    )

                    self.lbl_eta.config(
                        text=(
                            "預估剩餘："
                            f"{self._format_time(remain_sec)}"
                        )
                    )

            self.lbl_status.config(
                text=(
                    f"{self.matches_done:,}"
                    f" / "
                    f"{self.total_matches:,}"
                    f" 場"
                )
            )

        except Exception:

            pass

        self.root.after(
            1000,
            self._gui_tick
        )

    # ========================================================
    # 時間格式
    # ========================================================

    @staticmethod
    def _format_time(seconds):

        seconds = int(
            max(0, seconds)
        )

        hours = seconds // 3600

        minutes = (
            seconds % 3600
        ) // 60

        secs = seconds % 60

        if hours >= 24:

            days = hours // 24

            hours = hours % 24

            return (
                f"{days}天 "
                f"{hours}小時 "
                f"{minutes}分"
            )

        return (
            f"{hours}小時 "
            f"{minutes}分 "
            f"{secs}秒"
        )

    # ========================================================
    # 保存進度
    # ========================================================

    def _save_progress(self):

        try:

            data = {

                "matches_done":
                    self.matches_done,

                "total_matches":
                    self.total_matches,

                "timestamp":
                    time.time(),

                "overall":
                    self.overall,

                "h2h":
                    self.h2h
            }

            tmp_file = (
                PROGRESS_FILE
                + ".tmp"
            )

            with open(
                tmp_file,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    data,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

            os.replace(
                tmp_file,
                PROGRESS_FILE
            )

        except Exception as e:

            self.log(
                f"⚠️ 保存進度失敗：{e}"
            )

    # ========================================================
    # 保存 CSV
    # ========================================================

    def _save_csv_safe(self):

        try:

            # ----------------------------------------------
            # Overall
            # ----------------------------------------------

            rows = []

            for name, data in self.overall.items():

                played = data["Played"]

                rate = (
                    data["Wins"]
                    /
                    played
                    * 100
                    if played > 0
                    else 0
                )

                rows.append({
                    "Model": name,
                    "Played": played,
                    "Wins": data["Wins"],
                    "Losses": data["Losses"],
                    "Draws": data["Draws"],
                    "WinRate(%)": round(
                        rate,
                        2
                    )
                })

            df = pd.DataFrame(
                rows
            ).sort_values(
                by="WinRate(%)",
                ascending=False
            ).reset_index(
                drop=True
            )

            df.insert(
                0,
                "Rank",
                df.index + 1
            )

            tmp = (
                CSV_OVERALL
                + ".tmp"
            )

            df.to_csv(
                tmp,
                index=False,
                encoding="utf-8-sig"
            )

            os.replace(
                tmp,
                CSV_OVERALL
            )

            # ----------------------------------------------
            # Head-to-head
            # ----------------------------------------------

            h2h_rows = []

            for a in self.names:

                for b in self.names:

                    if a == b:

                        continue

                    stat = self.h2h[a][b]

                    total = (
                        stat["Wins"]
                        +
                        stat["Losses"]
                        +
                        stat["Draws"]
                    )

                    rate = (
                        stat["Wins"]
                        /
                        total
                        * 100
                        if total > 0
                        else 0
                    )

                    h2h_rows.append({

                        "Model_A": a,

                        "Model_B": b,

                        "Games": total,

                        "A_Wins":
                            stat["Wins"],

                        "B_Wins":
                            stat["Losses"],

                        "Draws":
                            stat["Draws"],

                        "A_WinRate_vs_B(%)":
                            round(
                                rate,
                                2
                            )
                    })

            tmp_h2h = (
                CSV_H2H
                + ".tmp"
            )

            pd.DataFrame(
                h2h_rows
            ).to_csv(
                tmp_h2h,
                index=False,
                encoding="utf-8-sig"
            )

            os.replace(
                tmp_h2h,
                CSV_H2H
            )

            self._save_progress()

        except Exception as e:

            self.log(
                f"⚠️ CSV 保存錯誤：{e}"
            )

    # ========================================================
    # 主比賽迴圈
    # ========================================================

    def _run_loop(self):

        all_pairs = self.pairs

        total_pairs = len(
            all_pairs
        )

        try:

            with ThreadPoolExecutor(
                max_workers=WORKERS
            ) as pool:

                for iteration in range(
                    TOTAL_ITERATIONS
                ):

                    if self.stop_event.is_set():

                        break

                    for chunk_start in range(
                        0,
                        total_pairs,
                        CHUNK_SIZE
                    ):

                        if self.stop_event.is_set():

                            break

                        self.pause_event.wait()

                        if self.stop_event.is_set():

                            break

                        chunk = all_pairs[
                            chunk_start:
                            chunk_start
                            + CHUNK_SIZE
                        ]

                        futures = {}

                        for b_name, w_name in chunk:

                            if self.stop_event.is_set():

                                break

                            future = pool.submit(
                                play_match,
                                b_name,
                                w_name,
                                self.valid_engines[b_name],
                                self.valid_engines[w_name]
                            )

                            futures[future] = (
                                b_name,
                                w_name
                            )

                        for future in as_completed(
                            futures
                        ):

                            if self.stop_event.is_set():
                                pass

                            b_name, w_name = (
                                futures[future]
                            )

                            try:

                                (
                                    _,
                                    _,
                                    winner,
                                    steps_record
                                ) = future.result(
                                    timeout=50
                                )

                            except Exception:

                                winner = "Draw"
                                steps_record = []

                            # 儲存每一局的每一步成 CSV 檔
                            if steps_record:
                                try:
                                    match_filename = f"match_{b_name}_vs_{w_name}_t{self.matches_done+1}_{int(time.time()*1000)}.csv"
                                    df_steps = pd.DataFrame(steps_record)
                                    df_steps.to_csv(
                                        os.path.join(MATCH_LOG_DIR, match_filename),
                                        index=False,
                                        encoding="utf-8-sig"
                                    )
                                except Exception:
                                    pass

                            # --------------------------------
                            # 更新統計
                            # --------------------------------

                            self.matches_done += 1

                            self.overall[
                                b_name
                            ]["Played"] += 1

                            self.overall[
                                w_name
                            ]["Played"] += 1

                            if winner == b_name:

                                self.overall[
                                    b_name
                                ]["Wins"] += 1

                                self.overall[
                                    w_name
                                ]["Losses"] += 1

                                self.h2h[
                                    b_name
                                ][
                                    w_name
                                ]["Wins"] += 1

                                self.h2h[
                                    w_name
                                ][
                                    b_name
                                ]["Losses"] += 1

                            elif winner == w_name:

                                self.overall[
                                    w_name
                                ]["Wins"] += 1

                                self.overall[
                                    b_name
                                ]["Losses"] += 1

                                self.h2h[
                                    w_name
                                ][
                                    b_name
                                ]["Wins"] += 1

                                self.h2h[
                                    b_name
                                ][
                                    w_name
                                ]["Losses"] += 1

                            else:

                                self.overall[
                                    b_name
                                ]["Draws"] += 1

                                self.overall[
                                    w_name
                                ]["Draws"] += 1

                                self.h2h[
                                    b_name
                                ][
                                    w_name
                                ]["Draws"] += 1

                                self.h2h[
                                    w_name
                                ][
                                    b_name
                                ]["Draws"] += 1

                            # --------------------------------
                            # Log
                            # --------------------------------

                            self.log(
                                f"第 "
                                f"{self.matches_done:6d}"
                                f"/"
                                f"{self.total_matches}"
                                f" | "
                                f"{b_name} vs {w_name}"
                                f" -> {winner}"
                            )

                            # --------------------------------
                            # 定期保存總表
                            # --------------------------------

                            if (
                                self.matches_done
                                %
                                BATCH_FLUSH
                                == 0
                            ):

                                self._save_csv_safe()

                                self.root.after(
                                    0,
                                    self._refresh_table
                                )

                                self.log(
                                    "💾 "
                                    f"已保存 "
                                    f"{self.matches_done:,} "
                                    "場結果"
                                )

                            if (
                                not
                                self.pause_event.is_set()
                            ):

                                self.pause_event.wait()

                        if self.stop_event.is_set():

                            break

                    if self.stop_event.is_set():

                        break

            self._save_csv_safe()

            self.root.after(
                0,
                self._refresh_table
            )

            if (
                self.matches_done
                >=
                self.total_matches
            ):

                self.log(
                    "🏁 全部對局完成！"
                )

            else:

                self.log(
                    "⏹ 比賽已停止，目前進度已保存。"
                )

        except Exception as e:

            self.log(
                f"❌ 主程式錯誤：{e}"
            )

            self._save_csv_safe()

        finally:

            self.is_running = False

            self.root.after(
                0,
                self._finish_ui
            )

    # ========================================================
    # 完成後 UI
    # ========================================================

    def _finish_ui(self):

        self.btn_start.config(
            state=tk.NORMAL
        )

        self.btn_pause.config(
            state=tk.DISABLED
        )

        self.btn_stop.config(
            state=tk.DISABLED
        )

        self._refresh_table()

    # ========================================================
    # 查看目前第一名
    # ========================================================

    def pick_champion(self):

        rows = []

        for name, data in self.overall.items():

            played = data["Played"]

            rate = (
                data["Wins"]
                /
                played
                *
                100
                if played > 0
                else 0
            )

            rows.append({
                "Model": name,
                "Played": played,
                "WinRate": rate
            })

        if not rows:

            return

        rows.sort(
            key=lambda x: x["WinRate"],
            reverse=True
        )

        best = rows[0]

        if best["Played"] == 0:

            messagebox.showinfo(
                "提示",
                "目前尚無對局資料。"
            )

            return

        best_exe = (
            self.valid_engines
            .get(
                best["Model"],
                "路徑未知"
            )
        )

        msg = (
            "🏆 目前勝率最高模型\n\n"
            f"模型：{best['Model']}\n"
            f"勝率：{best['WinRate']:.2f}%\n"
            f"對局：{best['Played']:,} 場\n\n"
            "執行檔：\n"
            f"{best_exe}"
        )

        self.log(
            f"👑 目前第一名："
            f"{best['Model']} "
            f"({best['WinRate']:.2f}%)"
        )

        messagebox.showinfo(
            "目前第一名",
            msg
        )


# ============================================================
# 6. 啟動
# ============================================================

if __name__ == "__main__":

    root = tk.Tk()

    app = FullTournamentGUI(
        root
    )

    root.mainloop()